In [3]:
# instalamos scikit-learn
!pip install scikit-learn scipy seaborn openpyxl

In [4]:
#Librerías 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

#Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

#Scipy: clustering jerárquico y dendrograma
from scipy.cluster.hierarchy import dendrogram, linkage

#Configuración general de gráficos (principios de Schwabish) 
plt.rcParams['figure.dpi']        = 130
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid']         = True
plt.rcParams['grid.linestyle']    = '--'
plt.rcParams['grid.alpha']        = 0.35
#pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)



In [5]:
#Rutas de los archivos
RUTA = r'/Users/nahuelvasquez/Documents/GitHub/BDyML---Grupo1-PRIVADO/TP3'

RUTA_2024 = RUTA + r'/usu_individual_T324.xlsx'
RUTA_2025 = RUTA + r'/usu_individual_T325.xlsx'

print('Rutas configuradas:')
print(f'  2024 T3: {RUTA_2024}')
print(f'  2025 T3: {RUTA_2025}')

Rutas configuradas:
  2024 T3: /Users/nahuelvasquez/Documents/GitHub/BDyML---Grupo1-PRIVADO/TP3/usu_individual_T324.xlsx
  2025 T3: /Users/nahuelvasquez/Documents/GitHub/BDyML---Grupo1-PRIVADO/TP3/usu_individual_T325.xlsx


In [6]:
#Carga de Bases 

eph_2024_raw = pd.read_excel(RUTA_2024)
print(f'  EPH 2024 T3 cargada: {eph_2024_raw.shape[0]:,} personas | {eph_2024_raw.shape[1]} variables')

eph_2025_raw = pd.read_excel(RUTA_2025)
print(f'  EPH 2025 T3 cargada: {eph_2025_raw.shape[0]:,} personas | {eph_2025_raw.shape[1]} variables')


FileNotFoundError: [Errno 2] No such file or directory: '/Users/nahuelvasquez/Documents/GitHub/BDyML---Grupo1-PRIVADO/TP3/usu_individual_T324.xlsx'

In [ ]:
# Continuación del trabajo realizado en el TP2. Seleccionamos las variables necesarias e incorporamos las adicionales requeridas por el TP3:
#- `CH12`, `CH13`, `CH14` → para construir la variable `educ` (años de educación)
#- `PP3E_TOT`, `PP3F_TOT` → para construir `horastrab` (horas totales trabajadas)
#- `CH03` → relación con el jefe de hogar (necesaria para filtrar `horastrab`)

# Variables a seleccionar
VARS = [
    # Identificación del hogar y persona
    'CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE', 'PONDERA',
    # Características personales
    'CH03', 'CH04', 'CH06', 'CH07', 'CH08',
    # Situación laboral y educativa
    'NIVEL_ED', 'ESTADO', 'CAT_OCUP',
    # Variables de formalidad
    'PP04C', 'PP04D_COD', 'PP07H',
    # Horas trabajadas
    'PP3E_TOT', 'PP3F_TOT',
    # Ingresos
    'P21', 'P47T',
    # Educación
    'CH12', 'CH13', 'CH14',
]

# Verificamos disponibilidad en cada base y filtramos
vars_2024 = [v for v in VARS if v in eph_2024_raw.columns]
vars_2025 = [v for v in VARS if v in eph_2025_raw.columns]

df_2024 = eph_2024_raw[vars_2024].copy()
df_2025 = eph_2025_raw[vars_2025].copy()

print(f'Variables seleccionadas — 2024: {len(vars_2024)} | 2025: {len(vars_2025)}')
print(f'Faltantes en 2024: {[v for v in VARS if v not in eph_2024_raw.columns]}')
print(f'Faltantes en 2025: {[v for v in VARS if v not in eph_2025_raw.columns]}')

In [ ]:
# Armonizamos los datos
# CH08 puede diferir entre años (int vs float)
# Usamos pd.to_numeric antes de Int64 para manejar cualquier residuo de texto
for df in [df_2024, df_2025]:
    if 'CH08' in df.columns:
        df['CH08'] = pd.to_numeric(df['CH08'], errors='coerce').astype('Int64')

# Verificamos que los tipos coincidan
tipos = pd.DataFrame({
    'Tipo_2024': df_2024.dtypes,
    'Tipo_2025': df_2025.dtypes
})
tipos['Coinciden'] = tipos['Tipo_2024'] == tipos['Tipo_2025']
inconsistencias = tipos[~tipos['Coinciden']]
print('Inconsistencias de tipo:', len(inconsistencias))
if len(inconsistencias) > 0:
    print(inconsistencias)

In [ ]:
# Limpieza de ingresos 
# El INDEC codifica 'no aplica' / 'no responde' con -9 en variables de ingreso.
# Reemplazamos cualquier valor negativo por NaN — no eliminamos la fila porque
# la persona sigue siendo válida para las demás variables.

for df in [df_2024, df_2025]:
    for var in ['P21', 'P47T']:
        if var in df.columns:
            df[var] = pd.to_numeric(df[var], errors='coerce')
            df[var] = df[var].where(df[var] >= 0, np.nan)

print('Mín P21  2024:', df_2024['P21'].min(), '| 2025:', df_2025['P21'].min())
print('Mín P47T 2024:', df_2024['P47T'].min(), '| 2025:', df_2025['P47T'].min())

In [ ]:
# Ajuste por inflación 
# Los pesos de 2024 T3 y 2025 T3 tienen distinto poder de compra
# Convertimos los ingresos de 2024 T3 a pesos de 2025 T3 usando el IPC del INDEC
# IPC base diciembre 2016 = 100:
#   2024 T3 (promedio jul-sep 2024) ≈ 6,872
#   2025 T3 (promedio jul-sep 2025) ≈ 9,201

IPC_2024T3 = 6871.0
IPC_2025T3 = 9200.4
factor = IPC_2025T3 / IPC_2024T3

print(f'Factor de actualización: {factor:.4f}')
print(f'Inflación acumulada 2024T3→2025T3: {(factor-1)*100:.1f}%')

# Aplicamos el factor a 2024 — creamos variables reales
df_2024['P21_real']  = df_2024['P21']  * factor
df_2024['P47T_real'] = df_2024['P47T'] * factor

# En 2025 las variables reales son iguales a las nominales
df_2025['P21_real']  = df_2025['P21']
df_2025['P47T_real'] = df_2025['P47T']

# Verificación correcta — filtramos P21 > 0 (no solo no-nulo)
verificacion = df_2024[df_2024['P21'] > 0][['P21', 'P21_real']].head(3).round(0)
print(verificacion)

# Y chequeamos que la relación sea siempre el factor
ratio = (df_2024['P21_real'] / df_2024['P21']).dropna()
ratio = ratio[ratio > 0]  # excluimos ceros

In [ ]:
#Etiqueta de año y concatenación
# ignore_index=True resetea el índice de 0 a N para evitar índices duplicados
# que causarían errores en operaciones posteriores de .loc

df_2024['año'] = 2024
df_2025['año'] = 2025

df_total = pd.concat([df_2024, df_2025], ignore_index=True)

print(f'Base total combinada: {len(df_total):,} personas')
print(f'Valores únicos de año: {sorted(df_total["año"].unique())}')

In [ ]:
#Indicador de informalidad (upper-tier)
#Es Asalariado, CAT_OCUP = 3
#Sin descuento Jubilatorio, PP07H = 2
#Empresa de más de 5 personas, PP04C ≥ 6

#Filtros de la EPH 
#ESTADO = 0 → no respondió condición de actividad → excluir
#ESTADO = 1 → ocupado → es el universo relevante para informalidad

respondieron = df_total[df_total['ESTADO'] != 0].copy()
ocupados_base = respondieron[respondieron['ESTADO'] == 1].copy()

print(f'Respondieron: {len(respondieron):,}')
print(f'Ocupados:     {len(ocupados_base):,}')

In [ ]:
def crear_indicador_informalidad(df):
    """
    Construye el indicador Upper-tier informal wage employee (Grupo 1).
    Retorna:
        1 = upper-tier informal (asalariado sin jubilación en empresa grande)
        0 = formal
        NaN = no aplica (no es asalariado o no responde)
    """
    df = df.copy()
    for col in ['CAT_OCUP', 'PP07H', 'PP04C']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    def clasificar(row):
        # Condición 1: debe ser asalariado
        if row.get('CAT_OCUP', np.nan) != 3:
            return np.nan
        # PP07H = 0 → no es asalariado según cuestionario → excluir
        if row.get('PP07H', np.nan) not in [1, 2]:
            return np.nan
        # PP07H = 1 → tiene descuento jubilatorio → formal
        if row.get('PP07H', np.nan) == 1:
            return 0
        # Condición 3: empresa de más de 5 personas (PP04C ≥ 6)
        pp04c = row.get('PP04C', np.nan)
        if pd.isna(pp04c) or pp04c == 99:
            return np.nan
        return 1 if pp04c >= 6 else 0

    df['informal'] = df.apply(clasificar, axis=1)
    return df

# Parte 1
## 1) Creacion de variables

In [ ]:
#1a. edad y edad2 
# CH06 = edad de la persona en años
# Filtramos edades negativas (código INDEC de no respuesta) y > 120 (imposibles)

respondieron['edad'] = pd.to_numeric(respondieron['CH06'], errors='coerce')
respondieron['edad'] = respondieron['edad'].where(
    (respondieron['edad'] >= 0) & (respondieron['edad'] <= 120), np.nan
)
respondieron['edad2'] = respondieron['edad'] ** 2

print('edad  — min:', respondieron['edad'].min(), '| max:', respondieron['edad'].max())
print('edad2 — min:', respondieron['edad2'].min(), '| max:', respondieron['edad2'].max())

In [ ]:
# 1b. educ: años de educación formal 
# Combina tres variables del cuestionario EPH:
#   CH12 = nivel educativo más alto alcanzado
#   CH13 = ¿completó ese nivel? (1=Sí, 2=No)
#   CH14 = último año aprobado dentro de ese nivel
#
# Años acumulados por nivel (sistema educativo argentino):
#   1=Jardín:0 base | 2=Primario:0 base,7 máx | 3=EGB:0 base,9 máx
#   4=Secundario:7 base,5 máx | 5=Polimodal:9 base,3 máx
#   6=Terciario:12 base,5 máx | 7=Universitario:12 base,5 máx
#   8=Post-universitario:17 base,5 máx

años_base = {1:0, 2:0, 3:0, 4:7, 5:9, 6:12, 7:12, 8:17, 9:0}
años_max  = {1:1, 2:7, 3:9, 4:5, 5:3, 6:5,  7:5,  8:5,  9:0}

def calcular_educ(row):
    try:
        nivel    = int(pd.to_numeric(row.get('CH12', 0), errors='coerce') or 0)
        completo = int(pd.to_numeric(row.get('CH13', 2), errors='coerce') or 2)
        ultimo   = pd.to_numeric(row.get('CH14', 0), errors='coerce')
        ultimo   = 0 if pd.isna(ultimo) else int(ultimo)
        base     = años_base.get(nivel, 0)
        maximo   = años_max.get(nivel, 0)
        if completo == 1:   # completó el nivel
            return base + maximo
        else:               # no completó → años cursados
            return base + min(ultimo, maximo)
    except:
        return np.nan

vars_educ = ['CH12', 'CH13', 'CH14']
if all(v in respondieron.columns for v in vars_educ):
    respondieron['educ'] = respondieron.apply(calcular_educ, axis=1)
    print('educ creada. Distribución por año:')
    print(respondieron.groupby('año')['educ'].describe().round(2))
else:
    respondieron['educ'] = np.nan
    print('Variables CH12/CH13/CH14 no disponibles')

In [ ]:
#1c. horastrab: horas totales trabajadas (solo jefe del hogar)
# PP3E_TOT = horas trabajadas en la ocupación principal
# PP3F_TOT = horas trabajadas en otras ocupaciones (puede ser 0)
# Solo se calcula para el jefe de hogar (CH03 == 1) como pide la consigna
# El código 999 en horas = no responde → se reemplaza por NaN

vars_h = ['PP3E_TOT', 'PP3F_TOT', 'CH03']
if all(v in respondieron.columns for v in vars_h):
    respondieron['PP3E_TOT'] = pd.to_numeric(respondieron['PP3E_TOT'], errors='coerce').replace(999, np.nan)
    respondieron['PP3F_TOT'] = pd.to_numeric(respondieron['PP3F_TOT'], errors='coerce').replace(999, np.nan).fillna(0)

    es_jefe = pd.to_numeric(respondieron['CH03'], errors='coerce') == 1
    respondieron['horastrab'] = np.where(
        es_jefe,
        respondieron['PP3E_TOT'] + respondieron['PP3F_TOT'],
        np.nan
    )
    print('horastrab creada. Distribución por año:')
    print(respondieron.groupby('año')['horastrab'].describe().round(2))
else:
    respondieron['horastrab'] = np.nan
    print('Variables de horas no disponibles')

In [ ]:
#1d. nhogar: número de miembros por hogar 
# Usamos CODUSU + NRO_HOGAR como identificador único del hogar
# y contamos cuántos COMPONENTE distintos tiene cada hogar.
# Hint de la consigna: usar CODUSU, NRO_HOGAR y COMPONENTE.

id_cols  = [c for c in ['CODUSU', 'NRO_HOGAR'] if c in respondieron.columns]
comp_col = 'COMPONENTE' if 'COMPONENTE' in respondieron.columns else None

if id_cols and comp_col:
    nhogar_df = (
        respondieron
        .groupby(id_cols)[comp_col]
        .count()
        .reset_index()
        .rename(columns={comp_col: 'nhogar'})
    )
    respondieron = respondieron.merge(nhogar_df, on=id_cols, how='left')
    print('nhogar creada. Distribución por año:')
    print(respondieron.groupby('año')['nhogar'].describe().round(2))
else:
    respondieron['nhogar'] = np.nan
    print('Variables de identificación de hogar no disponibles')

In [ ]:
# Reconstruimos ocupados con todas las variables nuevas 
# (reconstruimos ocupados DESPUÉS de crear las variables nuevas)
# en respondieron para garantizar que los índices estén alineados
# Luego aplicamos el indicador de informalidad

ocupados = respondieron[respondieron['ESTADO'] == 1].copy()
ocupados = crear_indicador_informalidad(ocupados)

# Asalariados con indicador definido → universo para PCA y Clustering
asal = ocupados[
    (ocupados['CAT_OCUP'] == 3) &
    (ocupados['informal'].notna())
].copy()

print(f'Ocupados:                  {len(ocupados):,}')
print(f'Asalariados con indicador: {len(asal):,}')
print()
print(asal['informal'].value_counts().rename({0: 'Formal', 1: 'Upper-tier informal'}))


## 2) Estadística Descriptiva

In [ ]:
# Variables para la tabla descriptiva
vars_desc = [
    'edad', 'edad2', 'educ', 'horastrab', 'nhogar',
    'P21', 'P47T', 'PP04C', 'NIVEL_ED',
    'CAT_OCUP', 'ESTADO', 'CH04', 'CH07', 'CH08'
]
vars_desc = [v for v in vars_desc if v in respondieron.columns]

for año in [2024, 2025]:
    sub  = respondieron[respondieron['año'] == año][vars_desc]
    desc = sub.describe(percentiles=[0.25, 0.50, 0.75])
    desc.index = ['N', 'Media', 'Desvío', 'Mín', 'P25', 'Mediana', 'P75', 'Máx']
    print(f'\n=== Estadística Descriptiva — {año} T3 ===')
    print(desc.round(2).to_string())


## 3) Matriz de Correlaciones



In [ ]:
#Matriz de correlaciones para 2025 T3 con las variables: `informal`, `edad`, `edad2`, `educ`, `horastrab`, `nhogar`, `P21`, `PP04C`
#Base: ocupados

vars_corr = ['informal', 'edad', 'edad2', 'educ', 'horastrab', 'nhogar', 'P21', 'PP04C']
vars_corr = [v for v in vars_corr if v in ocupados.columns]

corr_2025 = ocupados[ocupados['año'] == 2025][vars_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_2025,
    annot=True, fmt='.2f',
    cmap='RdBu', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 11}
)
ax.set_title(
    'Matriz de correlación — Ocupados EPH 2025 T3',
    fontsize=13, fontweight='bold', pad=12
)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.savefig(RUTA + r'\correlacion_2025.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: correlacion_2025.png')


## Parte 2: METODOS NO SUPERVISADOS
# A) PCA (Análisis de Componentes Principales)
# A.1) PCA


In [ ]:
#Variables: `edad`, `edad2`, `educ`, `horastrab`, `nhogar`, `P21`, `PP04C`  
#Base: asalariados con indicador de informalidad definido, año 2025 T3

# Variables para PCA
vars_pca = ['edad', 'edad2', 'educ', 'horastrab', 'nhogar', 'P21', 'PP04C']
vars_pca = [v for v in vars_pca if v in asal.columns]

# Subconjunto 2025 T3 sin nulos en ninguna variable de PCA
pca_data = asal[asal['año'] == 2025][vars_pca + ['informal']].dropna()

print(f'Observaciones para PCA: {len(pca_data):,}')
print(f'Variables incluidas ({len(vars_pca)}): {vars_pca}')

# Estandarizamos — CRÍTICO para PCA
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(pca_data[vars_pca])

# Aplicamos PCA con todos los componentes posibles
pca    = PCA(n_components=len(vars_pca))
scores = pca.fit_transform(X_scaled)

print('\nVarianza explicada por componente:')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {v*100:.1f}%  (acumulada: {sum(pca.explained_variance_ratio_[:i+1])*100:.1f}%)')

In [ ]:
## Gráfico de Scores
#Los scores son las coordenadas de cada persona en el nuevo sistema de ejes (componentes principales)
#Graficamos PC1 vs PC2 coloreando por condición de formalidad para ver si PCA logra separar naturalmente los dos grupos

colores   = {0: '#2980B9', 1: '#C0392B'}
etiquetas = {0: 'Formal', 1: 'Upper-tier Informal'}

fig, ax = plt.subplots(figsize=(10, 7))

for grupo, color in colores.items():
    mask = pca_data['informal'].values == grupo
    ax.scatter(
        scores[mask, 0], scores[mask, 1],
        c=color, label=etiquetas[grupo],
        alpha=0.35, s=15, linewidths=0
    )

ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.6, linestyle='--')
ax.set_xlabel(
    f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza explicada)', fontsize=12
)
ax.set_ylabel(
    f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza explicada)', fontsize=12
)
ax.set_title(
    'Scores PCA — PC1 vs PC2\nAsalariados EPH 2025 T3',
    fontsize=13, fontweight='bold', pad=12
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(RUTA + r'\pca_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: pca_scores.png')

## A.2) Gráfico de loadings

In [ ]:


loadings = pca.components_.T  # shape: (n_variables, n_componentes)

fig, ax = plt.subplots(figsize=(9, 7))

for i, var in enumerate(vars_pca):
    ax.arrow(
        0, 0,
        loadings[i, 0], loadings[i, 1],
        head_width=0.03, head_length=0.03,
        fc='#C0392B', ec='#C0392B', linewidth=2
    )
    ax.text(
        loadings[i, 0] * 1.18, loadings[i, 1] * 1.18,
        var, fontsize=11, ha='center', va='center',
        color='#1E2761', fontweight='bold'
    )

# Círculo unitario de referencia
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta),
        color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.35, 1.35)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax.set_title(
    'Loadings PCA — Ponderadores por componente\nAsalariados EPH 2025 T3',
    fontsize=13, fontweight='bold', pad=12
)
plt.tight_layout()
plt.savefig(RUTA + r'\pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: pca_loadings.png')

## A.3) Gráfico de la proporción de la varianza


In [ ]:
#El **scree plot** muestra cuánta información captura cada componente. Las barras muestran la varianza individual y la línea roja la acumulada. 
#Permite decidir cuántos componentes conservar (criterio usual: llegar al 80% acumulado)

varianza      = pca.explained_variance_ratio_ * 100
varianza_acum = np.cumsum(varianza)
componentes   = [f'PC{i+1}' for i in range(len(vars_pca))]

fig, ax1 = plt.subplots(figsize=(9, 5))

# Barras: varianza individual por componente
bars = ax1.bar(componentes, varianza, color='#2980B9', alpha=0.8, label='Varianza individual')
ax1.set_ylabel('Varianza explicada (%)', fontsize=12, color='#2980B9')
ax1.tick_params(axis='y', labelcolor='#2980B9')

# Línea: varianza acumulada (eje derecho)
ax2 = ax1.twinx()
ax2.plot(componentes, varianza_acum, color='#C0392B',
         marker='o', linewidth=2.5, markersize=7, label='Varianza acumulada')
ax2.axhline(80, color='gray', linewidth=1, linestyle='--', alpha=0.7)
ax2.text(len(vars_pca)-1.5, 81, '80%', fontsize=10, color='gray')
ax2.set_ylabel('Varianza acumulada (%)', fontsize=12, color='#C0392B')
ax2.tick_params(axis='y', labelcolor='#C0392B')
ax2.set_ylim(0, 108)

# Anotamos el % sobre cada barra
for bar, v in zip(bars, varianza):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f'{v:.1f}%', ha='center', va='bottom', fontsize=10)

ax1.set_title(
    'Scree Plot — Proporción de varianza explicada\nAsalariados EPH 2025 T3',
    fontsize=13, fontweight='bold', pad=12
)
ax1.set_xlabel('Componente Principal', fontsize=12)

l1, lab1 = ax1.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lab1+lab2, fontsize=10, loc='center right')

plt.tight_layout()
plt.savefig(RUTA + r'\pca_varianza.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: pca_varianza.png')


##B)Cluster

### B.4.a) K-Means con k=2

In [ ]:
# K-Means con k=2 y n_init=20 como pide la consigna
kmeans2    = KMeans(n_clusters=2, n_init=20, random_state=42)
kmeans2.fit(X_scaled)
labels_km  = kmeans2.labels_

# Gráfico comparativo: clusters encontrados vs informalidad real
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: clusters de K-Means
colores_km = {0: '#E67E22', 1: '#8E44AD'}
for grupo in [0, 1]:
    mask = labels_km == grupo
    axes[0].scatter(
        pca_data['edad'].values[mask],
        pca_data['P21'].values[mask],
        c=colores_km[grupo], label=f'Cluster {grupo+1}',
        alpha=0.35, s=15, linewidths=0
    )
axes[0].set_xlabel('Edad', fontsize=12)
axes[0].set_ylabel('Ingreso P21 (nominal)', fontsize=12)
axes[0].set_title('K-Means (k=2) — Clusters encontrados', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))

# Panel derecho: informalidad real (referencia)
for grupo, color in colores.items():
    mask = pca_data['informal'].values == grupo
    axes[1].scatter(
        pca_data['edad'].values[mask],
        pca_data['P21'].values[mask],
        c=color, label=etiquetas[grupo],
        alpha=0.35, s=15, linewidths=0
    )
axes[1].set_xlabel('Edad', fontsize=12)
axes[1].set_ylabel('Ingreso P21 (nominal)', fontsize=12)
axes[1].set_title('Informalidad real (referencia)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))

fig.suptitle(
    'K-Means k=2 vs Informalidad real — Asalariados EPH 2025 T3',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(RUTA + r'\kmeans_k2.png', dpi=150, bbox_inches='tight')
plt.show()

# Adjusted Rand Index: mide similitud entre clusters y etiqueta real
# 0 = agrupación aleatoria | 1 = coincidencia perfecta
ari = adjusted_rand_score(pca_data['informal'].values, labels_km)
print(f'Adjusted Rand Index (K-Means vs informalidad real): {ari:.3f}')
print('Guardado: kmeans_k2.png')

### B.4.b) Método del Codo (Elbow) para k=1 a k=40

In [ ]:
# Usamos una muestra para acelerar el cómputo con k grande
np.random.seed(42)
n_sample = min(5000, len(X_scaled))
idx      = np.random.choice(len(X_scaled), n_sample, replace=False)
X_elbow  = X_scaled[idx]

inercias = []
ks       = range(1, 41)
print(f'Corriendo K-Means k=1 a 40 sobre muestra de {n_sample:,} obs (puede tardar ~1 minuto)...')

for k in ks:
    km = KMeans(n_clusters=k, n_init=5, random_state=42, max_iter=100)
    km.fit(X_elbow)
    inercias.append(km.inertia_)
    if k % 10 == 0:
        print(f'  k={k} completado')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(list(ks), inercias, color='#1E2761', linewidth=2.5, marker='o', markersize=4)
ax.set_xlabel('Número de clusters (k)', fontsize=12)
ax.set_ylabel('Inercia (suma de dist. al centroide²)', fontsize=12)
ax.set_title(
    'Método del Codo (Elbow) — K-Means k=1 a k=40\nAsalariados EPH 2025 T3',
    fontsize=13, fontweight='bold', pad=12
)
plt.tight_layout()
plt.savefig(RUTA + r'\elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: elbow.png')


## B.5) Clustering Jerárquico y Dendrograma


In [ ]:
# Usamos una muestra de 200 obs — con miles de observaciones el dendrograma
# es ilegible porque cada hoja sería una persona
np.random.seed(42)
n_dendro = min(200, len(X_scaled))
idx_d    = np.random.choice(len(X_scaled), n_dendro, replace=False)
X_dendro = X_scaled[idx_d]

# Método Ward: minimiza la varianza intra-cluster en cada fusión
# Es el criterio más usado en práctica porque produce clusters compactos
Z = linkage(X_dendro, method='ward', metric='euclidean')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z, ax=ax,
    truncate_mode='lastp',    # muestra solo las últimas p fusiones (más legible)
    p=30,
    leaf_rotation=90,
    leaf_font_size=9,
    color_threshold=Z[-4, 2]  # colorea los grupos por encima del umbral
)

# Línea de corte sugerida
ax.axhline(y=Z[-4, 2], color='#C0392B', linewidth=1.5, linestyle='--', alpha=0.8)
ax.text(
    ax.get_xlim()[1]*0.72, Z[-4, 2]*1.04,
    'Corte sugerido (~4 clusters)',
    color='#C0392B', fontsize=10
)

ax.set_title(
    'Dendrograma — Clustering Jerárquico (Ward linkage)\nAsalariados EPH 2025 T3 (muestra n=200)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Observaciones (agrupadas)', fontsize=11)
ax.set_ylabel('Disimilitud (distancia euclidiana)', fontsize=11)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.savefig(RUTA + r'\dendrograma.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: dendrograma.png')


## Parte 3: Herramientas de IA y reflexión final

Claude nos ayudó durante el TP sobre todo en la parte de armonización de datos, limpieza de la base y armado de gráficos. Fue útil para ordenar el código, revisar cómo compatibilizar variables de distintos años de la EPH, detectar posibles errores y mejorar la forma de presentar los resultados visualmente. También nos sirvió para resolver dudas puntuales de programación. Sin embargo, lo más difícil fue usar la IA para interpretar los gráficos, especialmente en PCA y clustering, porque los resultados no se explican solos y requieren entender qué está pasando con los datos y relacionarlo con el contexto laboral de la EPH.

**Tiempo total dedicado: 8h-10h semanales 


## Prompts de IA utilizados




In [ ]:
#"hola, estoy haciendo un tp de big data con la EPH del indec y tengo que crear una variable que se llame educ que represente los años de educacion formal de cada persona. 
#tengo las variables CH12 CH13 y CH14 pero no entiendo bien como combinarlas, CH12 es el nivel educativo, CH13 si lo completo y CH14 el ultimo año aprobado. como armo eso en python? 
#el sistema educativo argentino tiene primaria de 7 años, secundario de 5 y despues universitario"


#"me esta dando este error cuando trato de propagar variables nuevas a mi dataframe de ocupados: KeyError: '[92505, 92507...] not in index'. 
#lo que hice fue crear variables nuevas en el dataframe respondieron y despues quise pasarlas a ocupados con .loc pero me da ese error.
#entiendo que tiene que ver con los indices pero no se como resolverlo"


#"una pregunta conceptual, en el scree plot que hice para el PCA me sale que necesito 5 componentes 
#para llegar al 80% de varianza explicada, y la varianza esta bastante distribuida entre los primeros 5 (29%, 21%, 16%, 14%, 12%).
#la consigna me pide comentar esto en relacion a las correlaciones que vi antes entre las variables.
#no entiendo bien la conexion entre las dos cosas, me podes explicar?"